In [1]:
"""Imports, constants, and session metadata."""


from __future__ import annotations

import contextlib
import gc
from datetime import datetime
from pathlib import Path
from typing import Final

import numpy as np
import polars as pl
from scipy.stats import t as t_dist
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

now = datetime.now()
print(f"Last notebook execution: {now}")

# ── Constants ──────────────────────────────────────────────────────────

_METRIC_KEYS: Final[list[str]] = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "specificity",
    "balanced_accuracy",
    "mcc",
    "eer",
    "brier_score",
    "ece",
    "log_loss",
]
_EPS: Final[float] = 1e-12

INPUT_PATHS: Final[list[str]] = [
    "../tables/all_predictions.parquet",
    "../tables/ensemble_predictions.parquet",
]
CHECKPOINT_DIR: Final[Path] = Path("../tables/checkpoints")
POOLED_PATH: Final[Path] = Path("../tables/metrics_pooled.parquet")
MACRO_PATH: Final[Path] = Path("../tables/metrics_macro.parquet")
OUTPUT_PATH: Final[Path] = Path("../tables/metrics.parquet")
SUMMARY_CSV: Final[Path] = Path("../tables/metrics_summary.csv")

COLUMNS: Final[list[str]] = [
    "path",
    "y_true",
    "y_pred",
    "y_proba_pos",
    "gaze_yaw",
    "gaze_pitch",
    "au01",
    "au02",
    "au03",
    "au04",
    "au05",
    "au06",
    "au07",
    "au08",
    "model",
    "run",
    "seed",
    "ctx",
    "subject_id",
]

Last notebook execution: 2026-11-09 04:56:37.874973


In [2]:
print("Input files:")
for p in INPUT_PATHS:
    path = Path(p)
    size_gb = path.stat().st_size / (1024**3)
    print(f"  {path.name}: {size_gb:.2f} GB")

Input files:
  all_predictions.parquet: 2.68 GB
  ensemble_predictions.parquet: 0.41 GB


In [3]:
# ═══════════════════════════════════════════════════════════════════
"""Metric helper functions: EER, ECE, pooled binary, and subject-level macro."""


def calculate_eer(y_true: np.ndarray, y_scores: np.ndarray) -> float:
    """EER via direct crossing of FPR and FNR curves.

    Args:
        y_true: Ground-truth labels, shape (n,).
        y_scores: Probability of the positive class, shape (n,).

    Returns:
        Equal Error Rate, or NaN if the crossing cannot be determined.
    """
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    diff = fpr - fnr

    if len(fpr) < 2 or np.all(diff > 0) or np.all(diff < 0):
        return np.nan

    idx = np.where(np.diff(np.sign(diff)))[0]
    if len(idx) == 0:
        return np.nan

    i = idx[0]
    x0, x1 = fpr[i], fpr[i + 1]
    y0, y1 = diff[i], diff[i + 1]
    return float(x0 - y0 * (x1 - x0) / (y1 - y0))


def calculate_ece(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    y_pred: np.ndarray,
    n_bins: int = 10,
) -> float:
    """Confidence-ECE (Guo et al. 2017).

    Args:
        y_true: Ground-truth labels, shape (n,).
        y_prob: Probability of the positive class, shape (n,).
        y_pred: Predicted labels, shape (n,).
        n_bins: Number of equal-width confidence bins.

    Returns:
        Expected Calibration Error in [0, 1].
    """
    confidence = np.where(y_pred == 1, y_prob, 1.0 - y_prob)
    correct = (y_true == y_pred).astype(float)

    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        if i == 0:
            in_bin = (confidence >= lo) & (confidence <= hi)
        else:
            in_bin = (confidence > lo) & (confidence <= hi)

        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            avg_confidence = confidence[in_bin].mean()
            avg_accuracy = correct[in_bin].mean()
            ece += abs(float(avg_accuracy) - float(avg_confidence)) * float(prop_in_bin)

    return ece


def _compute_binary_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_scores: np.ndarray,
) -> dict[str, float]:
    """Compute the full binary metric set for a single prediction array.

    Args:
        y_true: Ground-truth labels, shape (n,).
        y_pred: Predicted labels, shape (n,).
        y_scores: Probability of the positive class, shape (n,).

    Returns:
        Dict with one entry per metric in _METRIC_KEYS.
    """
    m: dict[str, float] = dict.fromkeys(_METRIC_KEYS, np.nan)
    y_scores_clipped = np.clip(y_scores, _EPS, 1.0 - _EPS)

    m["accuracy"] = float(accuracy_score(y_true, y_pred))
    m["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    m["recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    m["f1_score"] = float(f1_score(y_true, y_pred, zero_division=0))
    m["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    if len(np.unique(y_true)) >= 2:
        m["roc_auc"] = float(roc_auc_score(y_true, y_scores))
        m["eer"] = calculate_eer(y_true, y_scores)
        m["log_loss"] = float(log_loss(y_true, y_scores_clipped))

    if 0 in y_true:
        m["specificity"] = float(
            recall_score(y_true, y_pred, pos_label=0, zero_division=0)
        )

    with contextlib.suppress(ValueError, ZeroDivisionError):
        m["mcc"] = float(matthews_corrcoef(y_true, y_pred))

    m["brier_score"] = float(brier_score_loss(y_true, y_scores_clipped))
    m["ece"] = calculate_ece(y_true, y_scores, y_pred)

    return m


def _compute_subject_level(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_scores: np.ndarray,
    subject_ids: np.ndarray,
) -> dict[str, float]:
    """Compute macro (subject-level) metrics by averaging per-subject values.

    Uses sort + split instead of per-subject boolean masks.

    Args:
        y_true: Ground-truth labels, shape (n,).
        y_pred: Predicted labels, shape (n,).
        y_scores: Probability of the positive class, shape (n,).
        subject_ids: Subject identifier per sample, shape (n,).

    Returns:
        Dict with keys '<metric>_macro', one per metric in _METRIC_KEYS.
    """
    _, inverse = np.unique(subject_ids, return_inverse=True)
    order = np.argsort(inverse, kind="stable")
    sorted_inverse = inverse[order]
    split_points = np.where(np.diff(sorted_inverse) != 0)[0] + 1
    groups = np.split(order, split_points)

    per_subject: list[dict[str, float]] = [
        _compute_binary_metrics(y_true[g], y_pred[g], y_scores[g]) for g in groups
    ]

    macro: dict[str, float] = {}
    for key in _METRIC_KEYS:
        values = [s[key] for s in per_subject if not np.isnan(s[key])]
        macro[f"{key}_macro"] = float(np.mean(values)) if values else np.nan

    return macro

In [4]:
# ═══════════════════════════════════════════════════════════════════
"""Compute pooled and macro metrics per model, with parquet checkpointing."""

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Build the lazy frame once (stays in the query plan — no collect yet)
lf = pl.concat([pl.scan_parquet(p).select(COLUMNS) for p in INPUT_PATHS])

# Tiny query: only reads the "model" column + unique
models: list[str] = (
    lf.select("model").unique().sort("model").collect().to_series().to_list()
)
print(f"Models ({len(models)}): {models}")

for model in models:
    checkpoint = CHECKPOINT_DIR / f"{model}.parquet"
    if checkpoint.exists():
        print(f"  ⏭  {model} — checkpoint exists, skipping")
        continue

    print(f"  ▶  {model} ...", end=" ", flush=True)

    # Predicate pushdown: only this model's rows are materialized
    df: pl.DataFrame = lf.filter(pl.col("model") == model).collect()

    y_true_all = df["y_true"].to_numpy().astype(np.int32)
    y_pred_all = df["y_pred"].to_numpy().astype(np.int32)
    y_scores_all = df["y_proba_pos"].to_numpy().astype(np.float64)
    subject_all = df["subject_id"].to_numpy()
    seeds = df["seed"].to_numpy()
    ctxs = df["ctx"].to_numpy()

    # Partition by (seed, ctx)
    group_keys = np.array(list(zip(seeds, ctxs, strict=True)))
    unique_groups, inverse = np.unique(group_keys, axis=0, return_inverse=True)
    order = np.argsort(inverse, kind="stable")
    sorted_inverse = inverse[order]
    split_points = np.where(np.diff(sorted_inverse) != 0)[0] + 1
    group_slices = np.split(order, split_points)

    combined: list[dict[str, str | int | float]] = []

    for i, (seed, ctx) in enumerate(unique_groups):
        idx = group_slices[i]
        yt = y_true_all[idx]
        yp = y_pred_all[idx]
        ys = y_scores_all[idx]

        row: dict[str, str | int | float] = {
            "model": model,
            "seed": int(seed),
            "ctx": int(ctx),
            **_compute_binary_metrics(yt, yp, ys),
            **_compute_subject_level(yt, yp, ys, subject_all[idx]),
        }
        combined.append(row)

    pl.DataFrame(combined).write_parquet(checkpoint)
    print(f"done ({len(combined)} rows)")

    # Free memory before next model
    del df, y_true_all, y_pred_all, y_scores_all, subject_all
    del seeds, ctxs, group_keys, order, sorted_inverse, split_points, group_slices
    gc.collect()

print("All models processed.")

Models (7): ['causilo', 'ensemble', 'kumo', 'lightgbm', 'tabdpt', 'tabicl', 'tabpfn']
  ⏭  causilo — checkpoint exists, skipping
  ⏭  ensemble — checkpoint exists, skipping
  ⏭  kumo — checkpoint exists, skipping
  ⏭  lightgbm — checkpoint exists, skipping
  ⏭  tabdpt — checkpoint exists, skipping
  ⏭  tabicl — checkpoint exists, skipping
  ⏭  tabpfn — checkpoint exists, skipping
All models processed.


In [5]:
# ═══════════════════════════════════════════════════════════════════
"""Merge per-model checkpoints into the final pooled, macro, and combined tables."""

checkpoint_files = sorted(CHECKPOINT_DIR.glob("*.parquet"))
if not checkpoint_files:
    raise FileNotFoundError(f"No checkpoints found in {CHECKPOINT_DIR}")

all_metrics = pl.concat([pl.scan_parquet(p) for p in checkpoint_files]).collect()

# Split into pooled and macro views
pooled_cols = ["model", "seed", "ctx", *_METRIC_KEYS]
macro_cols = ["model", "seed", "ctx", *[f"{k}_macro" for k in _METRIC_KEYS]]

pooled_df = all_metrics.select(pooled_cols)
pooled_df.write_parquet(POOLED_PATH)
print(f"Saved {pooled_df.height} rows → {POOLED_PATH}")

macro_df = all_metrics.select(macro_cols)
macro_df.write_parquet(MACRO_PATH)
print(f"Saved {macro_df.height} rows → {MACRO_PATH}")

all_metrics.write_parquet(OUTPUT_PATH)
print(f"Saved {all_metrics.height} rows → {OUTPUT_PATH}")
print(f"Columns: {len(all_metrics.columns)}")

Saved 350 rows → ../tables/metrics_pooled.parquet
Saved 350 rows → ../tables/metrics_macro.parquet
Saved 350 rows → ../tables/metrics.parquet
Columns: 27


In [6]:
# ═══════════════════════════════════════════════════════════════════
"""Summary table (descriptive metrics): mean and std across seeds per model × ctx."""

SUMMARY_DESC_CSV: Final[Path] = Path("../tables/metrics_summary_descriptive.csv")

summary_desc = (
    all_metrics.group_by(["model", "ctx"])
    .agg(
        pl.col("balanced_accuracy").mean().alias("bal_acc_mean"),
        pl.col("balanced_accuracy").std().alias("bal_acc_std"),
        pl.col("specificity").mean().alias("spec_mean"),
        pl.col("specificity").std().alias("spec_std"),
        pl.col("roc_auc").mean().alias("auc_mean"),
        pl.col("roc_auc").std().alias("auc_std"),
        pl.col("f1_score").mean().alias("f1_mean"),
        pl.col("f1_score").std().alias("f1_std"),
        pl.col("eer").mean().alias("eer_mean"),
        pl.col("eer").std().alias("eer_std"),
        pl.col("balanced_accuracy_macro").mean().alias("bal_acc_macro_mean"),
        pl.col("balanced_accuracy_macro").std().alias("bal_acc_macro_std"),
        pl.col("specificity_macro").mean().alias("spec_macro_mean"),
        pl.col("specificity_macro").std().alias("spec_macro_std"),
        pl.col("roc_auc_macro").mean().alias("auc_macro_mean"),
        pl.col("roc_auc_macro").std().alias("auc_macro_std"),
        pl.col("f1_score_macro").mean().alias("f1_macro_mean"),
        pl.col("f1_score_macro").std().alias("f1_macro_std"),
    )
    .sort(["model", "ctx"])
)

summary_desc.write_csv(SUMMARY_DESC_CSV, separator=";")
print(f"Saved → {SUMMARY_DESC_CSV}")
print(summary_desc)

Saved → ../tables/metrics_summary_descriptive.csv
shape: (35, 20)
┌─────────┬─────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ model   ┆ ctx ┆ bal_acc_mea ┆ bal_acc_st ┆ … ┆ auc_macro_ ┆ auc_macro_ ┆ f1_macro_m ┆ f1_macro_s │
│ ---     ┆ --- ┆ n           ┆ d          ┆   ┆ mean       ┆ std        ┆ ean        ┆ td         │
│ str     ┆ i64 ┆ ---         ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│         ┆     ┆ f64         ┆ f64        ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞═════════╪═════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ causilo ┆ 5   ┆ 0.562069    ┆ 0.01496    ┆ … ┆ 0.636124   ┆ 0.024181   ┆ 0.509215   ┆ 0.02187    │
│ causilo ┆ 13  ┆ 0.570923    ┆ 0.007365   ┆ … ┆ 0.635743   ┆ 0.012797   ┆ 0.507444   ┆ 0.010695   │
│ causilo ┆ 26  ┆ 0.574337    ┆ 0.013201   ┆ … ┆ 0.643561   ┆ 0.012721   ┆ 0.503512   ┆ 0.016942   │
│ causilo ┆ 40  ┆ 0.59867

In [7]:
# ═══════════════════════════════════════════════════════════════════
"""Summary table (calibration metrics): mean and std across seeds per model × ctx."""

SUMMARY_CAL_CSV: Final[Path] = Path("../tables/metrics_summary_calibration.csv")

summary_cal = (
    all_metrics.group_by(["model", "ctx"])
    .agg(
        pl.col("brier_score").mean().alias("brier_mean"),
        pl.col("brier_score").std().alias("brier_std"),
        pl.col("log_loss").mean().alias("log_loss_mean"),
        pl.col("log_loss").std().alias("log_loss_std"),
        pl.col("ece").mean().alias("ece_mean"),
        pl.col("ece").std().alias("ece_std"),
        pl.col("brier_score_macro").mean().alias("brier_macro_mean"),
        pl.col("brier_score_macro").std().alias("brier_macro_std"),
        pl.col("log_loss_macro").mean().alias("log_loss_macro_mean"),
        pl.col("log_loss_macro").std().alias("log_loss_macro_std"),
        pl.col("ece_macro").mean().alias("ece_macro_mean"),
        pl.col("ece_macro").std().alias("ece_macro_std"),
    )
    .sort(["model", "ctx"])
)

summary_cal.write_csv(SUMMARY_CAL_CSV, separator=";")
print(f"Saved → {SUMMARY_CAL_CSV}")
print(summary_cal)

Saved → ../tables/metrics_summary_calibration.csv
shape: (35, 14)
┌─────────┬─────┬────────────┬───────────┬───┬─────────────┬─────────────┬────────────┬────────────┐
│ model   ┆ ctx ┆ brier_mean ┆ brier_std ┆ … ┆ log_loss_ma ┆ log_loss_ma ┆ ece_macro_ ┆ ece_macro_ │
│ ---     ┆ --- ┆ ---        ┆ ---       ┆   ┆ cro_mean    ┆ cro_std     ┆ mean       ┆ std        │
│ str     ┆ i64 ┆ f64        ┆ f64       ┆   ┆ ---         ┆ ---         ┆ ---        ┆ ---        │
│         ┆     ┆            ┆           ┆   ┆ f64         ┆ f64         ┆ f64        ┆ f64        │
╞═════════╪═════╪════════════╪═══════════╪═══╪═════════════╪═════════════╪════════════╪════════════╡
│ causilo ┆ 5   ┆ 0.322046   ┆ 0.031427  ┆ … ┆ 0.955039    ┆ 0.115784    ┆ 0.247101   ┆ 0.046024   │
│ causilo ┆ 13  ┆ 0.30259    ┆ 0.017502  ┆ … ┆ 0.859248    ┆ 0.043929    ┆ 0.211383   ┆ 0.027884   │
│ causilo ┆ 26  ┆ 0.284332   ┆ 0.012021  ┆ … ┆ 0.798305    ┆ 0.023421    ┆ 0.174355   ┆ 0.017856   │
│ causilo ┆ 40  ┆ 0.26609

In [8]:
# ═══════════════════════════════════════════════════════════════════
"""95% CI (Student's t) per model × ctx, using seeds as the sample.

Two tables: descriptive metrics and calibration metrics.
"""

CI_DESC_CSV: Final[Path] = Path("../tables/metrics_ci95_descriptive.csv")
CI_CAL_CSV: Final[Path] = Path("../tables/metrics_ci95_calibration.csv")

ci_desc_metrics: Final[list[str]] = [
    "balanced_accuracy",
    "specificity",
    "roc_auc",
    "f1_score",
    "eer",
    "balanced_accuracy_macro",
    "specificity_macro",
    "roc_auc_macro",
    "f1_score_macro",
    "eer_macro",
]
ci_cal_metrics: Final[list[str]] = [
    "brier_score",
    "log_loss",
    "ece",
    "brier_score_macro",
    "log_loss_macro",
    "ece_macro",
]

# Group by (model, ctx): compute mean, sample std, and count of seeds
grouped = (
    all_metrics.group_by(["model", "ctx"])
    .agg(
        pl.len().alias("n_seeds"),
        *[
            pl.col(m).mean().alias(f"{m}_mean")
            for m in ci_desc_metrics + ci_cal_metrics
        ],
        *[pl.col(m).std().alias(f"{m}_std") for m in ci_desc_metrics + ci_cal_metrics],
    )
    .sort(["model", "ctx"])
)

# Verify: print seed count per model × ctx
print("Seeds per model × ctx:")
print(grouped.select(["model", "ctx", "n_seeds"]))


def _compute_ci(
    grouped_df: pl.DataFrame,
    metrics: list[str],
) -> pl.DataFrame:
    """Compute 95% t-based CI bounds for a list of metrics.

    Args:
        grouped_df: DataFrame with '<metric>_mean', '<metric>_std', 'n_seeds'.
        metrics: Metric column names (without suffixes).

    Returns:
        DataFrame with model, ctx, n_seeds, and '<metric>_ci_lower/upper'.
    """
    rows: list[dict[str, str | int | float]] = []
    for row in grouped_df.iter_rows(named=True):
        n = row["n_seeds"]
        out: dict[str, str | int | float] = {
            "model": row["model"],
            "ctx": row["ctx"],
            "n_seeds": n,
        }
        for m in metrics:
            mean = row[f"{m}_mean"]
            std = row[f"{m}_std"]
            if n < 2 or std is None or np.isnan(std):
                out[f"{m}_ci_lower"] = np.nan
                out[f"{m}_ci_upper"] = np.nan
            else:
                t_crit = t_dist.ppf(0.975, df=n - 1)
                margin = t_crit * std / np.sqrt(n)
                out[f"{m}_ci_lower"] = mean - margin
                out[f"{m}_ci_upper"] = mean + margin
        rows.append(out)
    return pl.DataFrame(rows)


# Descriptive
ci_desc_df = _compute_ci(grouped, ci_desc_metrics)
ci_desc_df.write_csv(CI_DESC_CSV, separator=";")
print(f"Saved → {CI_DESC_CSV}")
print(ci_desc_df)

# Calibration
ci_cal_df = _compute_ci(grouped, ci_cal_metrics)
ci_cal_df.write_csv(CI_CAL_CSV, separator=";")
print(f"Saved → {CI_CAL_CSV}")
print(ci_cal_df)

Seeds per model × ctx:
shape: (35, 3)
┌─────────┬─────┬─────────┐
│ model   ┆ ctx ┆ n_seeds │
│ ---     ┆ --- ┆ ---     │
│ str     ┆ i64 ┆ u32     │
╞═════════╪═════╪═════════╡
│ causilo ┆ 5   ┆ 10      │
│ causilo ┆ 13  ┆ 10      │
│ causilo ┆ 26  ┆ 10      │
│ causilo ┆ 40  ┆ 10      │
│ causilo ┆ 47  ┆ 10      │
│ …       ┆ …   ┆ …       │
│ tabpfn  ┆ 5   ┆ 10      │
│ tabpfn  ┆ 13  ┆ 10      │
│ tabpfn  ┆ 26  ┆ 10      │
│ tabpfn  ┆ 40  ┆ 10      │
│ tabpfn  ┆ 47  ┆ 10      │
└─────────┴─────┴─────────┘
Saved → ../tables/metrics_ci95_descriptive.csv
shape: (35, 23)
┌─────────┬─────┬─────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬────────────┐
│ model   ┆ ctx ┆ n_seeds ┆ balanced_ac ┆ … ┆ f1_score_ma ┆ f1_score_ma ┆ eer_macro_c ┆ eer_macro_ │
│ ---     ┆ --- ┆ ---     ┆ curacy_ci_l ┆   ┆ cro_ci_lowe ┆ cro_ci_uppe ┆ i_lower     ┆ ci_upper   │
│ str     ┆ i64 ┆ i64     ┆ ower        ┆   ┆ r           ┆ r           ┆ ---         ┆ ---        │
│         ┆     ┆  

#### **Other**
_____

In [17]:
"""Check how many subjects have both classes at ensemble/ctx=47."""

lf = pl.concat([pl.scan_parquet(p).select(COLUMNS) for p in INPUT_PATHS])
grp = (
    lf.filter((pl.col("model") == "ensemble") & (pl.col("ctx") == 47))
    .select(["subject_id", "y_true"])
    .collect()
)

subject_classes = grp.group_by("subject_id").agg(
    pl.col("y_true").n_unique().alias("n_classes")
)
print(
    f"Subjects with 2 classes: {(subject_classes['n_classes'] == 2).sum()} / {len(subject_classes)}"
)

Subjects with 2 classes: 7 / 8


In [18]:
"""Per-seed: how many subjects have both classes at ensemble/ctx=47."""

grp = (
    lf.filter((pl.col("model") == "ensemble") & (pl.col("ctx") == 47))
    .select(["seed", "subject_id", "y_true"])
    .collect()
)

per_seed = (
    grp.group_by(["seed", "subject_id"])
    .agg(pl.col("y_true").n_unique().alias("n_classes"))
    .group_by("seed")
    .agg(
        pl.len().alias("n_subjects"),
        (pl.col("n_classes") == 2).sum().alias("n_with_both"),
    )
    .sort("seed")
)
print(per_seed)

shape: (7, 3)
┌──────┬────────────┬─────────────┐
│ seed ┆ n_subjects ┆ n_with_both │
│ ---  ┆ ---        ┆ ---         │
│ i64  ┆ u32        ┆ u32         │
╞══════╪════════════╪═════════════╡
│ 42   ┆ 1          ┆ 1           │
│ 69   ┆ 1          ┆ 0           │
│ 2512 ┆ 1          ┆ 1           │
│ 2607 ┆ 1          ┆ 1           │
│ 2691 ┆ 2          ┆ 1           │
│ 3141 ┆ 2          ┆ 2           │
│ 3312 ┆ 1          ┆ 1           │
└──────┴────────────┴─────────────┘


In [19]:
"""Check what subject_id actually contains."""

lf = pl.concat([pl.scan_parquet(p).select(COLUMNS) for p in INPUT_PATHS])

# How many unique subject_ids?
n_subjects = lf.select(pl.col("subject_id").n_unique()).collect().item()
print(f"Unique subject_ids: {n_subjects}")

# Show a few values
print(lf.select(["path", "subject_id"]).head(5).collect())

# Check for nulls
n_nulls = lf.select(pl.col("subject_id").is_null().sum()).collect().item()
print(f"Null subject_ids: {n_nulls}")

Unique subject_ids: 53
shape: (5, 2)
┌─────────────────────────────────┬────────────┐
│ path                            ┆ subject_id │
│ ---                             ┆ ---        │
│ str                             ┆ str        │
╞═════════════════════════════════╪════════════╡
│ /home/jovyan/projects/Vision-M… ┆ 9txq       │
│ /home/jovyan/projects/Vision-M… ┆ 9txq       │
│ /home/jovyan/projects/Vision-M… ┆ 9txq       │
│ /home/jovyan/projects/Vision-M… ┆ 9txq       │
│ /home/jovyan/projects/Vision-M… ┆ 9txq       │
└─────────────────────────────────┴────────────┘
Null subject_ids: 0


In [20]:
"""Compare subject counts: individual models vs ensemble at ctx=47."""

# Individual models (pick one)
indiv = (
    lf.filter((pl.col("model") == "tabpfn") & (pl.col("ctx") == 47))
    .select(["seed", "subject_id"])
    .collect()
)

print("Individual (tabpfn) subjects per seed at ctx=47:")
print(
    indiv.group_by("seed")
    .agg(pl.col("subject_id").n_unique().alias("n_subjects"))
    .sort("seed")
)

# Ensemble
ens = (
    lf.filter((pl.col("model") == "ensemble") & (pl.col("ctx") == 47))
    .select(["seed", "subject_id"])
    .collect()
)

print("\nEnsemble subjects per seed at ctx=47:")
print(
    ens.group_by("seed")
    .agg(pl.col("subject_id").n_unique().alias("n_subjects"))
    .sort("seed")
)

# Which subjects are missing from ensemble?
indiv_subjects = set(indiv["subject_id"].to_list())
ens_subjects = set(ens["subject_id"].to_list())
missing = indiv_subjects - ens_subjects
print(
    f"\nSubjects in tabpfn but NOT in ensemble: {len(missing)} / {len(indiv_subjects)}"
)
print(sorted(missing))

Individual (tabpfn) subjects per seed at ctx=47:
shape: (10, 2)
┌──────┬────────────┐
│ seed ┆ n_subjects │
│ ---  ┆ ---        │
│ i64  ┆ u32        │
╞══════╪════════════╡
│ 42   ┆ 6          │
│ 69   ┆ 6          │
│ 666  ┆ 6          │
│ 1112 ┆ 6          │
│ 1911 ┆ 6          │
│ 2512 ┆ 6          │
│ 2607 ┆ 6          │
│ 2691 ┆ 6          │
│ 3141 ┆ 6          │
│ 3312 ┆ 6          │
└──────┴────────────┘

Ensemble subjects per seed at ctx=47:
shape: (7, 2)
┌──────┬────────────┐
│ seed ┆ n_subjects │
│ ---  ┆ ---        │
│ i64  ┆ u32        │
╞══════╪════════════╡
│ 42   ┆ 1          │
│ 69   ┆ 1          │
│ 2512 ┆ 1          │
│ 2607 ┆ 1          │
│ 2691 ┆ 2          │
│ 3141 ┆ 2          │
│ 3312 ┆ 1          │
└──────┴────────────┘

Subjects in tabpfn but NOT in ensemble: 26 / 34
['45lx', '4woj', '5f7t', '6g6y', '6k5f', '71i5', '7h5u', '8g4y', '8i4i', '9j3o', '9t6n', '9txq', 'b2l8', 'bfl5', 'ctzy', 'cxj0', 'h8r2', 'h8s1', 'j9h8', 'kkf5', 'p9i3', 'qw5t', 't6v9', 'uymz', 'x1

In [ ]:
"""Check which TFM has no predictions.csv for ctx=47, seeds 666/1112/1911."""


results_dir = Path("../results")
missing_seeds = [666, 1112, 1911]

for model in ["tabpfn", "tabicl", "tabdpt", "causilo", "kumo"]:
    for seed in missing_seeds:
        folder = results_dir / f"{model}_results"
        # Find the run folder
        run_dirs = list(folder.glob("run_*")) if folder.exists() else []
        for run_dir in run_dirs:
            target = run_dir / model / f"seed_{seed}_ctx_47"
            if not target.exists():
                print(f"MISSING: {model} / seed_{seed}_ctx_47")

In [22]:
"""Row counts per model × seed at ctx=47."""

lf = pl.concat([pl.scan_parquet(p).select(COLUMNS) for p in INPUT_PATHS])

counts = (
    lf.filter(pl.col("ctx") == 47)
    .group_by(["model", "seed"])
    .agg(pl.len().alias("n_rows"))
    .sort(["model", "seed"])
)
print(counts)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SORT BY [col("model"), col("seed")]
  AGGREGATE[maintain_order: false]
    [len().alias("n_rows")] BY [col("model"), col("seed")]
    FROM
    FILTER col("ctx") == 47
    FROM
      UNION[maintain_order: true]
        PLAN 0:
          SELECT [col("path"), col("y_true"), col("y_pred"), col("y_proba_pos"), col("gaze_yaw"), col("gaze_pitch"), col("au01"), col("au02"), col("au03"), col("au04"), col("au05"), col("au06"), col("au07"), col("au08"), col("model"), col("run"), col("seed"), col("ctx"), col("subject_id")]
            Parquet SCAN [../tables/all_predictions.parquet]
            PROJECT */19 COLUMNS
            ESTIMATED ROWS: 40585398
        PLAN 1:
          SELECT [col("path"), col("y_true"), col("y_pred"), col("y_proba_pos"), col("gaze_yaw"), col("gaze_pitch"), col("au01"), col("au02"), col("au03"), col("au04"), col("au05"), col("au06"), col("au07"), col("au08"), col("model"), col("run"), col("seed"

In [27]:
df = (
    lf.group_by(["model", "seed", "ctx"])
    .agg(pl.len().alias("n_rows"))
    .sort(["model", "ctx", "seed"])
    .collect()
)
print(df.to_pandas().to_string())

        model  seed  ctx  n_rows
0     causilo    42    5  241784
1     causilo    69    5  241739
2     causilo   666    5  241813
3     causilo  1112    5  241765
4     causilo  1911    5  241859
5     causilo  2512    5  241765
6     causilo  2607    5  242383
7     causilo  2691    5  241742
8     causilo  3141    5  241795
9     causilo  3312    5  244265
10    causilo    42   13  205104
11    causilo    69   13  205250
12    causilo   666   13  200332
13    causilo  1112   13  204693
14    causilo  1911   13  200807
15    causilo  2512   13  205131
16    causilo  2607   13  202054
17    causilo  2691   13  200120
18    causilo  3141   13  200217
19    causilo  3312   13  200294
20    causilo    42   26  135917
21    causilo    69   26  133379
22    causilo   666   26  135954
23    causilo  1112   26  139453
24    causilo  1911   26  137154
25    causilo  2512   26  132811
26    causilo  2607   26  134037
27    causilo  2691   26  138730
28    causilo  3141   26  137852
29    caus

In [28]:
"""Path overlap at ctx=47, seed=42."""

paths = {}
for m in ["tabpfn", "tabicl", "tabdpt", "causilo", "kumo"]:
    p = (
        lf.filter(
            (pl.col("model") == m) & (pl.col("seed") == 42) & (pl.col("ctx") == 47)
        )
        .select("path")
        .collect()
        .to_series()
        .to_list()
    )
    paths[m] = set(p)
    print(f"{m}: {len(p):,} paths")

overlap_all = set.intersection(*paths.values())
print(f"\nAll 5 overlap: {len(overlap_all):,}")

overlap_no_tabicl = set.intersection(
    paths["tabpfn"], paths["tabdpt"], paths["causilo"], paths["kumo"]
)
print(f"4 (without tabicl): {len(overlap_no_tabicl):,}")

tabpfn: 31,151 paths
tabicl: 29,379 paths
tabdpt: 31,151 paths
causilo: 31,151 paths
kumo: 31,151 paths

All 5 overlap: 5,199
4 (without tabicl): 31,145


In [29]:
"""Compare the actual path sets — are tabicl's paths a SUBSET, or a DIFFERENT set?"""

tabpfn_paths = set(
    lf.filter(
        (pl.col("model") == "tabpfn") & (pl.col("seed") == 42) & (pl.col("ctx") == 47)
    )
    .select("path")
    .collect()
    .to_series()
    .to_list()
)
tabicl_paths = set(
    lf.filter(
        (pl.col("model") == "tabicl") & (pl.col("seed") == 42) & (pl.col("ctx") == 47)
    )
    .select("path")
    .collect()
    .to_series()
    .to_list()
)

print(f"tabpfn paths: {len(tabpfn_paths):,}")
print(f"tabicl paths: {len(tabicl_paths):,}")
print(f"tabicl ⊂ tabpfn? {tabicl_paths.issubset(tabpfn_paths)}")
print(f"tabpfn ⊂ tabicl? {tabpfn_paths.issubset(tabicl_paths)}")
print(f"Only in tabpfn: {len(tabpfn_paths - tabicl_paths):,}")
print(f"Only in tabicl: {len(tabicl_paths - tabpfn_paths):,}")

tabpfn paths: 31,145
tabicl paths: 29,379
tabicl ⊂ tabpfn? False
tabpfn ⊂ tabicl? False
Only in tabpfn: 25,946
Only in tabicl: 24,180


In [30]:
subjects = np.array(["a", "b", "c", "d", "e", "f", "g", "h", "i", "j"])

# Old API
np.random.seed(42)
old_choice = np.random.choice(subjects, size=5, replace=False)

# New API
new_choice = np.random.default_rng(42).choice(subjects, size=5, replace=False)

print(f"Old (RandomState): {old_choice}")
print(f"New (Generator):   {new_choice}")

Old (RandomState): ['i' 'b' 'f' 'a' 'h']
New (Generator):   ['f' 'd' 'h' 'a' 'e']
